In [ ]:
import os
import glob
import re
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.dialects.postgresql import insert
from dotenv import load_dotenv

In [ ]:
DATA_DIR = "data/air"
load_dotenv() 
DB_URL = os.getenv("DB_URL")

NAME_NORMALIZATION = {
    "Bolbes": "Volvi",
    "Ampelokepon-Menemenes": "Ampelokipoi-Menemeni",
    "Ampelokipon": "Ampelokipoi-Menemeni",
    "Thermaikou": "Thermaikos",
    "Khalkedonos": "Chalkidona",
    "Chalkidonos": "Chalkidona",
    "Kordeliou": "Kordelio-Evosmos",
    "Kordeliou-Euosmou": "Kordelio-Evosmos",
    "Lagkada": "Lagkadas",
    "Neapoles-Sykeon": "Neapoli-Sykies",
    "Neapoles-Sukeon": "Neapoli-Sykies",       
    "Neapoli": "Neapoli-Sykies",
    "Pavlou_Mela": "Pavlos Melas",
    "Paulou_Mela": "Pavlos Melas",             
    "Thermes": "Thermi",
    "Thessalonikes": "Thessaloniki",
    "Pulaia": "Pylaia-Chortiatis",             
    "Pulaias-Khortiate": "Pylaia-Chortiatis",   
    "Oraiokastrou": "Oraiokastro"               
}

In [ ]:
file_pattern = os.path.join(DATA_DIR, "*", "municipality_of_*_pollutants_conc_timeseries-yearly_*.csv")
all_files = glob.glob(file_pattern)

if not all_files:
    print(f"No CSV files found matching the pattern in '{DATA_DIR}' subfolders!")

all_daily_summaries = []

for file_path in all_files:
    filename = os.path.basename(file_path)
    
    match = re.search(r"municipality_of_(.+)_pollutants_conc_timeseries-yearly", filename)
    if not match:
        continue
        
    raw_municipality = match.group(1).title()
    municipality = NAME_NORMALIZATION.get(raw_municipality, raw_municipality)
    
    df = pd.read_csv(file_path)
    
    #Normalizes column names
    df.rename(columns={
        'co_conc': 'co',
        'no2_conc': 'no2',
        'so2_conc': 'so2',
        'o3_conc': 'o3'
    }, inplace=True)
    
    #Strips the hours/minutes to get a clean daily date
    df['time'] = pd.to_datetime(df['time'])
    df['date'] = df['time'].dt.date
    
    #Groups by date and calculates the daily mean for each particle
    daily_summary = df.groupby(['date']).agg({
        'no2': 'mean',
        'o3': 'mean',
        'co': 'mean',
        'so2': 'mean'
    }).reset_index()
    
    daily_summary = daily_summary.round(2)
    daily_summary['municipality'] = municipality
    
    #Reorders columns to match the db model
    final_df = daily_summary[['municipality', 'date', 'no2', 'o3', 'co', 'so2']]
    
    all_daily_summaries.append(final_df)

In [ ]:
if all_daily_summaries:
    master_df = pd.concat(all_daily_summaries, ignore_index=True)
    
    #Drops rows where a day had no data for any of the 4 particles
    master_df.dropna(subset=['no2', 'o3', 'co', 'so2'], how='all', inplace=True)
    
    print(f"Ready to insert {len(master_df)} daily records into the database.")

In [ ]:
def insert_do_nothing(table, conn, keys, data_iter):
    data = [dict(zip(keys, row)) for row in data_iter]
    
    insert_stmt = insert(table.table).values(data)

    do_nothing_stmt = insert_stmt.on_conflict_do_nothing(
        index_elements=['municipality', 'date']
    )
    
    result = conn.execute(do_nothing_stmt)
    return result.rowcount

sync_db_url = DB_URL.replace("+asyncpg", "")

engine = create_engine(sync_db_url)

master_df.to_sql(
    'historical_particles', 
    engine, 
    if_exists='append', 
    index=False, 
    method=insert_do_nothing
)